# H3 Editor on a Colab GPU

Runs ComfyUI with [H3 Editor](https://github.com/igorls/comfyui-h3-editor) on a Colab GPU, for renders your own machine
cannot fit: finals above all, which need far more memory than drafts.

**Before you run it:** pick a GPU runtime (Runtime, Change runtime type). The default models need about 40 GB of GPU
memory for a full-size final; a larger card renders faster and leaves room.

Run the cells from the top. The models are about 55 GB and download once per runtime, in roughly ten minutes. A
runtime's disk is gone when the runtime is released, so either keep the runtime alive while you work, or turn on the
Drive cache below so the next runtime copies the models back instead of downloading them.

Model files come under their own licenses; read each model's page before you use it.

In [ ]:
#@title 1. Settings, and the GPU this runtime has
#@markdown Keep a copy of the models on Google Drive, so a new runtime does not download them again (needs about 55 GB of Drive space).
MODELS_ON_DRIVE = False  #@param {type:"boolean"}
#@markdown Which ComfyUI to install: a tag such as `v0.45.0`, or `master`.
COMFYUI_REF = "v0.45.0"  #@param {type:"string"}
#@markdown Which H3 Editor to install: a tag, a commit or `main`.
EDITOR_REF = "main"  #@param {type:"string"}

import os, subprocess, sys
ROOT = "/content/ComfyUI"
PACK = ROOT + "/custom_nodes/comfyui-h3-editor"
DRIVE_CACHE = "/content/drive/MyDrive/h3-editor-models"
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
      or "No GPU: pick a GPU runtime first (Runtime, Change runtime type).")

In [ ]:
#@title 2. Install ComfyUI and H3 Editor
import os, subprocess, sys

def sh(*command):
    print("$", " ".join(command))
    subprocess.run(list(command), check=True)

def said(*command):
    return subprocess.run(list(command), capture_output=True, text=True).stdout.strip()

if not os.path.isdir(ROOT):
    sh("git", "clone", "--quiet", "https://github.com/comfyanonymous/ComfyUI", ROOT)
    if COMFYUI_REF != "master":
        sh("git", "-C", ROOT, "checkout", "--quiet", COMFYUI_REF)
if not os.path.isdir(PACK):
    sh("git", "clone", "--quiet", "https://github.com/igorls/comfyui-h3-editor", PACK)
    if EDITOR_REF != "main":
        sh("git", "-C", PACK, "checkout", "--quiet", EDITOR_REF)
sh(sys.executable, "-m", "pip", "install", "-q", "-r", ROOT + "/requirements.txt")
sh(sys.executable, "-m", "pip", "install", "-q", "-r", PACK + "/requirements.txt", "hf_transfer")
print("ComfyUI", said("git", "-C", ROOT, "describe", "--tags", "--always"), "| H3 Editor", said("git", "-C", PACK, "log", "-1", "--format=%h"))

In [ ]:
#@title 3. Get the models (about 55 GB, once per runtime)
# The files the default profile renders with, and the two a final adds. Each is found in its repository by name.
MODELS = [
    ("Comfy-Org/MiniMax-H3", "qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors", "text_encoders"),
    ("Comfy-Org/MiniMax-H3", "minimax_h3_video_vae_fp16.safetensors", "vae"),
    ("Comfy-Org/MiniMax-H3", "minimax_h3_audio_vae_fp32.safetensors", "vae"),
    ("VideoRebirthAI/hyperflow-h3-flash", "minimax_h3_ref2va_hyperflow_flash_14b-int8_convrot.safetensors", "diffusion_models/hyperflow"),
    ("VideoRebirthAI/hyperflow-h3-flash", "minimax_h3_fl2va_hyperflow_flash_14b-int8_convrot.safetensors", "diffusion_models/hyperflow"),
    ("LBH-123-AI/Minimax_h3_latent_Upscaler", "minimax_h3_latent_upscaler_3d_conv_v1_bf16.safetensors", "latent_upscale_models"),
    ("OzzyGT/taeh3", "taeh3.safetensors", "vae_approx"),
]
import os, shutil, time
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
from huggingface_hub import HfApi, hf_hub_download

if MODELS_ON_DRIVE and not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")
listed = {}
for repo, name, folder in MODELS:
    target = os.path.join(ROOT, "models", folder, name)
    os.makedirs(os.path.dirname(target), exist_ok=True)
    kept = os.path.join(DRIVE_CACHE, folder, name)
    if os.path.exists(target):
        print("have      ", name)
        continue
    began = time.time()
    if MODELS_ON_DRIVE and os.path.exists(kept):
        shutil.copyfile(kept, target)
        print("from Drive", name, "%.0f s" % (time.time() - began))
        continue
    if repo not in listed:
        listed[repo] = HfApi().list_repo_files(repo)
    inside = next((f for f in listed[repo] if f.split("/")[-1] == name), None)
    if not inside:
        print("NOT FOUND ", name, "in", repo, "- see docs/models.md in the pack for where it lives now")
        continue
    got = hf_hub_download(repo, inside, local_dir="/content/hf")
    shutil.move(got, target)
    print("downloaded", name, "%.1f GB in %.0f s" % (os.path.getsize(target) / 1e9, time.time() - began))
    if MODELS_ON_DRIVE:
        os.makedirs(os.path.dirname(kept), exist_ok=True)
        shutil.copyfile(target, kept)
print("Free disk: %.0f GB" % (shutil.disk_usage("/content").free / 1e9))

In [ ]:
#@title 4. Bring a project (optional)
#@markdown An archive made on your own machine: a `.tar` holding `h3take_projects/<project>/...` as it sits in ComfyUI's
#@markdown output folder, and optionally `models/...` files such as a LoRA. Give its path here (a file you dropped into
#@markdown the Files pane on the left lands in `/content`), or leave it empty to start with no project.
ARCHIVE = "/content/h3_project.tar"  #@param {type:"string"}
import os, tarfile
if ARCHIVE and os.path.exists(ARCHIVE):
    with tarfile.open(ARCHIVE) as tar:
        for member in tar.getmembers():
            top = member.name.split("/")[0]
            if top == "h3take_projects":
                tar.extract(member, ROOT + "/output", filter="data")
            elif top == "models":
                tar.extract(member, ROOT, filter="data")
            else:
                print("skipped", member.name)
    print("Projects here:", sorted(os.listdir(ROOT + "/output/h3take_projects")) if os.path.isdir(ROOT + "/output/h3take_projects") else [])
else:
    print("No archive at", ARCHIVE, "- continuing with no project.")

In [ ]:
#@title 5. Start ComfyUI and open the editor
import os, subprocess, sys, time, urllib.request
if not globals().get("COMFY") or COMFY.poll() is not None:
    LOG = open("/content/comfyui.log", "a")
    # --disable-cuda-malloc: with PyTorch's async CUDA allocator, a second final in a row ended the whole process
    # ("CUDA error: invalid argument" from cuMemFreeAsync) instead of freeing memory and carrying on.
    COMFY = subprocess.Popen([sys.executable, "main.py", "--listen", "127.0.0.1", "--port", "8188", "--disable-cuda-malloc"],
                             cwd=ROOT, stdout=LOG, stderr=subprocess.STDOUT)
for _ in range(180):
    try:
        urllib.request.urlopen("http://127.0.0.1:8188/h3take/api/projects", timeout=3)
        break
    except Exception:
        if COMFY.poll() is not None:
            raise SystemExit("ComfyUI stopped. The end of its log:\n" + open("/content/comfyui.log").read()[-3000:])
        time.sleep(2)
else:
    raise SystemExit("ComfyUI did not answer in six minutes. See /content/comfyui.log")
print("ComfyUI is up. The editor opens in a new tab, for you alone:")
from google.colab import output
output.serve_kernel_port_as_window(8188, path="/h3take/editor")

In [ ]:
#@title 6. Render clips
#@markdown The project and the clips to render as finals, by name, separated by `|`. Each is rendered and waited for in
#@markdown turn, so a clip that extends another is named after it. Leave CLIPS empty to list the project's clips.
PROJECT = "My_project"  #@param {type:"string"}
CLIPS = ""  #@param {type:"string"}
QUALITY = "final"  #@param ["final", "draft"]
import json, time, urllib.request
API = "http://127.0.0.1:8188/h3take/api"

def call(method, path, body=None, timeout=3600):
    data = None if body is None else json.dumps(body).encode()
    request = urllib.request.Request(API + path, data, {"Content-Type": "application/json"}, method=method)
    try:
        return json.load(urllib.request.urlopen(request, timeout=timeout))
    except urllib.error.HTTPError as error:
        raise SystemExit("%s %s: %s" % (method, path, error.read().decode()[:1500]))

project = call("GET", "/project/" + PROJECT)
if not CLIPS.strip():
    for clip in project["clips"]:
        print(clip["name"], "|", (clip["takes"][-1]["quality"] if clip["takes"] else "not rendered"))
for name in [n.strip() for n in CLIPS.split("|") if n.strip()]:
    clip = next((c for c in call("GET", "/project/" + PROJECT)["clips"] if c["name"] == name), None)
    if not clip:
        print(name, "| no such clip")
        continue
    began = time.time()
    reply = call("POST", "/project/%s/ops" % PROJECT, {"ops": [{"op": "render", "clip": clip["id"], "quality": QUALITY}]})
    if reply.get("error"):
        print(name, "|", reply["error"])
        continue
    call("GET", "/project/%s/wait?timeout=1800" % PROJECT)
    clip = next(c for c in call("GET", "/project/" + PROJECT)["clips"] if c["id"] == clip["id"])
    take = next((t for t in clip["takes"] if t["id"] == clip.get("active")), None)
    done = take and take.get("quality") == QUALITY and not clip.get("error")
    print(name, "|", "%s, %d frames, %s" % (take["file"], take["frames"], "x".join(map(str, take.get("size") or []))) if done else clip.get("error"),
          "| %.0f s" % (time.time() - began))

In [ ]:
#@title 7. Take the project home
#@markdown Packs the project as it is now, with every take, into one file you can download from the Files pane or that
#@markdown is copied to Drive. Put it back in your own ComfyUI's `output` folder to carry on there.
TO_DRIVE = False  #@param {type:"boolean"}
import os, shutil, tarfile
packed = "/content/%s_from_colab.tar" % PROJECT
with tarfile.open(packed, "w") as tar:
    tar.add(os.path.join(ROOT, "output", "h3take_projects", PROJECT), arcname="h3take_projects/" + PROJECT)
print(packed, "%.0f MB" % (os.path.getsize(packed) / 1e6))
if TO_DRIVE:
    if not os.path.isdir("/content/drive/MyDrive"):
        from google.colab import drive
        drive.mount("/content/drive")
    shutil.copyfile(packed, "/content/drive/MyDrive/" + os.path.basename(packed))
    print("Copied to Drive.")

## Notes

- **Memory.** A draft needs little; a full-size final needs the most. If a final fails for memory, set the project's
  Final size to Large in the editor's Project tab: it skips the enlarging pass.
- **Logs.** ComfyUI writes to `/content/comfyui.log`. `!tail -50 /content/comfyui.log` shows the end of it.
- **The editor link** from step 5 works only in your own browser, signed in to this Colab. Nothing is exposed publicly.
- **VFX clips** render in the browser, so they render wherever the editor page is open, this one included.